# Surrogate Model Ensemble Analysis

After training a surrogate ensemble with `cleo-optimize-train`, this notebook walks through key diagnostics to assess model quality before using it for downstream optimization.

We cover:
1. **Training curves** — did training converge? Is the model overfitting?
2. **Prediction accuracy** — how well does the ensemble mean predict held-out data?
3. **Uncertainty calibration** — does predicted variance correlate with actual error?
4. **Mu–sigma correlation** — are variance estimates independent of the mean?
5. **Uncertainty vs. distance from training set** — does the model know what it doesn't know?
6. **Top-k accuracy** — can the model rank the best sequences correctly?

This example uses a MoMI enzyme surrogate trained on 4 rounds of experimental data (~3.4k sequences).

In [ ]:
from pathlib import Path

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from omegaconf import OmegaConf
from tqdm.auto import tqdm

from cleo.optimize.utils.ensemble import Ensemble
from cleo.optimize.utils.train_data import SequenceFunctionDataset
from plot_utils import setup_style, CLEO_PALETTE, PRIMARY, SECONDARY, COLOR_REFERENCE, COLOR_THRESHOLD

setup_style()
%matplotlib inline

In [ ]:
REPO_ROOT = Path("../").resolve()
EXAMPLE_DIR = REPO_ROOT / "example_data" / "surrogate_model"

ckpt_path = EXAMPLE_DIR / "last.ckpt"
config_path = EXAMPLE_DIR / "config.yaml"
metrics_path = EXAMPLE_DIR / "csv_logs" / "metrics.csv"
dataset_path = EXAMPLE_DIR / "dataset.csv"

print(f"Checkpoint:  {ckpt_path}")
print(f"Config:      {config_path}")
print(f"Metrics:     {metrics_path}")
print(f"Dataset:     {dataset_path}")

In [ ]:
config = OmegaConf.load(config_path)

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
model = Ensemble(config)
model.load_state_dict(ckpt["state_dict"])
model.eval()

print(f"Ensemble of {config.model.num_models} models")
print(f"Architecture: {config.model.base_model.model_type}, hidden_dim={config.model.base_model.hidden_dim}")
print(f"Loss: NLL weight={config.model.loss.nll_weight}, MSE weight={config.model.loss.mse_weight}")

## 1. Training Curves

First, check that training has converged and there is no severe overfitting. We look at:
- **Validation NLL** — the primary loss; lower is better.
- **Pearson/Spearman correlation** — how well the model ranks sequences on the held-out set.
- **Per-model loss spread** — if individual ensemble members diverge, this may indicate training instability.

In [ ]:
metrics = pd.read_csv(metrics_path)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(metrics["epoch"], metrics["val/nll"], linewidth=1.5, color=PRIMARY)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Validation NLL")
axes[0].set_title("Validation NLL")

axes[1].plot(metrics["epoch"], metrics["val/pearsonr"], label="Pearson r", linewidth=1.5, color=PRIMARY)
axes[1].plot(metrics["epoch"], metrics["val/spearmanr"], label="Spearman ρ", linewidth=1.5, color=SECONDARY)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Correlation")
axes[1].set_title("Validation Correlation")
axes[1].legend()

nll_cols = [c for c in metrics.columns if c.startswith("val/nll_model_")]
for i, col in enumerate(nll_cols):
    axes[2].plot(metrics["epoch"], metrics[col], alpha=0.5, linewidth=0.8, color=CLEO_PALETTE[i % len(CLEO_PALETTE)])
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("NLL")
axes[2].set_title("Per-Model Validation NLL")

plt.tight_layout()
plt.show()

print(f"Final Pearson r:  {metrics['val/pearsonr'].iloc[-1]:.3f}")
print(f"Final Spearman ρ: {metrics['val/spearmanr'].iloc[-1]:.3f}")
print(f"Final val NLL:    {metrics['val/nll'].iloc[-1]:.3f}")

### Top-k Accuracy Over Training

Top-k accuracy measures the overlap between the model's predicted top-k sequences and the true top-k. This is often the most practically relevant metric — we care most about correctly identifying the best sequences.

In [ ]:
topk_cols = [c for c in metrics.columns if c.startswith("val/top")]

fig, ax = plt.subplots(figsize=(8, 4))
for i, col in enumerate(topk_cols):
    k = col.split("/")[1].replace("_accuracy", "")
    ax.plot(metrics["epoch"], metrics[col], label=k, linewidth=1.5, color=CLEO_PALETTE[i % len(CLEO_PALETTE)])

ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Top-k Accuracy on Validation Set")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 2. Prediction Accuracy

Now we load the full dataset and run the ensemble in inference mode to get predicted mean (μ) and standard deviation (σ) for each sequence. We plot ground truth vs. predicted mean on the validation set to visualize accuracy.

In [ ]:
data = pd.read_csv(dataset_path)
print(f"Total sequences: {len(data)}")
print(f"Train: {(~data['val']).sum()}, Val: {data['val'].sum()}")
print(f"Rounds: {sorted(data['round_number'].unique())}")
print(f"Label column: {config.data.dataset_cfg.label_col}")

In [ ]:
val_data = data[data["val"]].reset_index(drop=True)
val_dataset = SequenceFunctionDataset(config.data.dataset_cfg, val_data)

y_true, mu_list, sigma_list = [], [], []

with torch.no_grad():
    for x, y in tqdm(val_dataset, desc="Running predictions"):
        output = model(x.unsqueeze(0))
        y_true.append(y.item())
        mu_list.append(output["mu"].item())
        sigma_list.append(output["sigma"].item())

y_true = np.array(y_true)
mu = np.array(mu_list)
sigma = np.array(sigma_list)

In [ ]:
r_pearson, _ = stats.pearsonr(y_true, mu)
r_spearman, _ = stats.spearmanr(y_true, mu)

fig, ax = plt.subplots(figsize=(6, 6), dpi=100)
ax.scatter(y_true, mu, alpha=0.4, s=15, color=PRIMARY, edgecolor="white", linewidth=0.3)
lims = [min(y_true.min(), mu.min()) - 0.3, max(y_true.max(), mu.max()) + 0.3]
ax.plot(lims, lims, "--", color=COLOR_REFERENCE, linewidth=0.8)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Ground Truth")
ax.set_ylabel("Predicted Mean (μ)")
ax.set_title(f"Validation: Pearson r={r_pearson:.3f}, Spearman ρ={r_spearman:.3f}")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 3. Uncertainty Calibration

A well-calibrated ensemble should predict higher variance for sequences it gets wrong. We check this by plotting **squared error** vs. **predicted standard deviation**. A positive correlation means the model's uncertainty is informative.

This is important for acquisition function optimization — if the variance estimates are meaningless, UCB-based selection will not outperform greedy selection by the mean alone.

In [ ]:
se = (y_true - mu) ** 2
corr_se_sigma, _ = stats.pearsonr(se, sigma)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=100)

axes[0].scatter(se, sigma, alpha=0.4, s=15, color=PRIMARY, edgecolor="white", linewidth=0.3)
axes[0].set_xlabel("Squared Error")
axes[0].set_ylabel("Predicted Std (σ)")
axes[0].set_title(f"SE vs. σ  |  Pearson r={corr_se_sigma:.3f}")

corr_y_sigma, _ = stats.pearsonr(y_true, sigma)
axes[1].scatter(y_true, sigma, alpha=0.4, s=15, color=SECONDARY, edgecolor="white", linewidth=0.3)
axes[1].set_xlabel("Ground Truth")
axes[1].set_ylabel("Predicted Std (σ)")
axes[1].set_title(f"True Value vs. σ  |  Pearson r={corr_y_sigma:.3f}")

plt.tight_layout()
plt.show()

## 4. Mu–Sigma Correlation

If σ is strongly correlated with μ, the variance estimates may just be a proxy for the mean magnitude rather than genuine epistemic uncertainty. In that case, UCB will behave similarly to ranking by mean alone. Ideally σ captures uncertainty independently of the predicted value.

In [ ]:
corr_mu_sigma, _ = stats.pearsonr(mu, sigma)

fig, ax = plt.subplots(figsize=(6, 5), dpi=100)
ax.scatter(mu, sigma, alpha=0.4, s=15, color=PRIMARY, edgecolor="white", linewidth=0.3)
ax.set_xlabel("Predicted Mean (μ)")
ax.set_ylabel("Predicted Std (σ)")
ax.set_title(f"μ vs. σ  |  Pearson r={corr_mu_sigma:.3f}")
plt.tight_layout()
plt.show()

## 5. Uncertainty vs. Distance from Training Set

We expect the model to be more uncertain about sequences that are far from anything it has seen during training. To check this, we compute each validation sequence's minimum Hamming distance to any training sequence and plot it against σ.

A positive trend here means the ensemble is properly expressing higher uncertainty in under-explored regions of sequence space — exactly the behavior we want for guiding exploration.

In [ ]:
train_seqs = data[~data["val"]]["sequence"].tolist()
val_seqs = val_data["sequence"].tolist()

train_arr = np.array([list(s) for s in train_seqs])

min_dists = []
for seq in tqdm(val_seqs, desc="Computing Hamming distances"):
    seq_arr = np.array(list(seq))
    dists = (train_arr != seq_arr).sum(axis=1)
    min_dists.append(dists.min())

min_dists = np.array(min_dists)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=100)

sns.stripplot(x=min_dists, y=sigma, orient="h", alpha=0.2, size=3, ax=axes[0])
axes[0].set_xlabel("Min Hamming Distance to Training Set")
axes[0].set_ylabel("Predicted Std (σ)")
axes[0].set_title("Uncertainty vs. Novelty")

sns.boxplot(x=min_dists, y=se, orient="h", ax=axes[1], fliersize=2)
axes[1].set_xlabel("Min Hamming Distance to Training Set")
axes[1].set_ylabel("Squared Error")
axes[1].set_title("Error vs. Novelty")

plt.tight_layout()
plt.show()

## 6. Top-k Accuracy

In practice, we use the surrogate to select a batch of sequences to test experimentally. The key question is: **does the model correctly identify the best sequences?**

Top-k accuracy measures what fraction of the model's predicted top-k overlap with the true top-k. We compute this at several values of k.

In [ ]:
k_values = [5, 10, 20, 40, 60, 80, 100]
accuracies = []

y_t = torch.tensor(y_true)
mu_t = torch.tensor(mu)

for k in k_values:
    if k > len(y_true):
        break
    _, gt_idx = y_t.topk(k)
    _, pred_idx = mu_t.topk(k)
    overlap = torch.isin(pred_idx, gt_idx).sum().item() / k
    accuracies.append(overlap)
    print(f"Top-{k:>3d} accuracy: {overlap:.1%}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(k) for k in k_values[:len(accuracies)]], accuracies, color=PRIMARY, edgecolor="white")
ax.set_xlabel("k")
ax.set_ylabel("Accuracy")
ax.set_title("Top-k Accuracy (Validation Set)")
ax.set_ylim(0, 1)
for i, v in enumerate(accuracies):
    ax.text(i, v + 0.02, f"{v:.0%}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## 7. Per-Round Breakdown

Since this dataset spans multiple experimental rounds, it is useful to check whether the model performs consistently across rounds or if it struggles with sequences from certain rounds (e.g., later rounds with more diverse or higher-activity sequences).

In [ ]:
val_data_with_preds = val_data.copy()
val_data_with_preds["mu"] = mu
val_data_with_preds["sigma"] = sigma
val_data_with_preds["se"] = se

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=100)

for i, (rnd, grp) in enumerate(val_data_with_preds.groupby("round_number")):
    r, _ = stats.pearsonr(grp[config.data.dataset_cfg.label_col], grp["mu"])
    axes[0].scatter(
        grp[config.data.dataset_cfg.label_col], grp["mu"],
        alpha=0.4, s=15, label=f"Round {rnd} (r={r:.2f}, n={len(grp)})",
        color=CLEO_PALETTE[i % len(CLEO_PALETTE)], edgecolor="white", linewidth=0.3,
    )

lims = [min(y_true.min(), mu.min()) - 0.3, max(y_true.max(), mu.max()) + 0.3]
axes[0].plot(lims, lims, "--", color=COLOR_REFERENCE, linewidth=0.8)
axes[0].set_xlabel("Ground Truth")
axes[0].set_ylabel("Predicted Mean (μ)")
axes[0].set_title("Prediction Accuracy by Round")
axes[0].legend(fontsize=8)

sns.boxplot(data=val_data_with_preds, x="round_number", y="se", ax=axes[1], fliersize=2)
axes[1].set_xlabel("Experimental Round")
axes[1].set_ylabel("Squared Error")
axes[1].set_title("Error Distribution by Round")

plt.tight_layout()
plt.show()

## Summary

**Key things to look for:**

| Diagnostic | Good sign | Warning sign |
|---|---|---|
| Training curves | Smooth convergence, no divergence | Loss still decreasing at end, or val loss increasing |
| Pearson/Spearman r | > 0.8 for ranking-based tasks | < 0.6 may need more data or different architecture |
| SE vs. σ correlation | Positive (> 0.2) | Near zero or negative — variance is uninformative |
| μ–σ correlation | Weak | Strong correlation — σ is just a proxy for μ |
| σ vs. distance | σ increases with distance | No relationship — model overconfident on novel sequences |
| Top-k accuracy | Increases with k | Flat or decreasing — model ranking is poor |

If variance estimates are poorly calibrated, consider:
- Using MSE loss instead of (or in addition to) NLL
- Increasing ensemble size for better variance estimates
- Adding regularization to prevent overconfident models
- Relying on mean-based ranking instead of UCB for acquisition